In [5]:
# 3. Self-Reflection Prompt for Improving Output
#Goal: Ask the AI to critique and improve its summary.


import os
from dataclasses import dataclass

OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "").strip()
_HAS_OPENAI = False
try:
    if OPENAI_API_KEY:
        from openai import OpenAI
        client = OpenAI()
        _HAS_OPENAI = True
except Exception:
    _HAS_OPENAI = False

@dataclass
class LLMResponse:
    text: str

class MockLLM:
    def chat(self, prompt: str) -> LLMResponse:
        p = prompt.lower()
        if "generate python code" in p:
            code = (
                "def unique_sorted_squares(nums):\n"
                "    return sorted({n*n for n in nums})\n"
                "\n"
                "print(unique_sorted_squares([3,-1,2,2,-3,0]))"
            )
            return LLMResponse(code)
        if "summarize" in p and "reflect" not in p:
            return LLMResponse("Summary: LLMs can hallucinate; clear benchmarks and tool use reduce risk.")
        if "critique" in p or "reflect" in p:
            return LLMResponse("Critique: Add one concrete example and an evaluation best practice.")
        if "improved summary" in p:
            return LLMResponse("Improved Summary: LLMs may hallucinate, but using retrieval, human review, and clear benchmarks mitigates risks.")
        return LLMResponse("OK")

class OpenAILLM:
    def chat(self, prompt: str) -> LLMResponse:
        resp = client.responses.create(
            model="gpt-4o-mini",
            input=[{"role":"user","content":prompt}],
            temperature=0.3,
        )
        out = []
        for o in getattr(resp, "output", []):
            if getattr(o, "type", "") == "output_text":
                out.append(o.text)
        return LLMResponse("\n".join(out) if out else str(resp))

LLM = OpenAILLM() if _HAS_OPENAI else MockLLM()
print("Using OpenAI:", _HAS_OPENAI)


Using OpenAI: False


In [4]:
# ✅ Prompt 3: Self-Reflection Prompt for Improving Output
# This version fixes the "Improved Summary" issue and works with both MockLLM and OpenAI.

# Make sure you've already run the LLM setup cell above before running this one.

# Passage to summarize
passage = '''Large language models (LLMs) are powerful but can hallucinate when data is missing.
Proper evaluation requires clear benchmarks and carefully designed prompts.
Teams can reduce risk by adding tool use, retrieval, and human review for critical decisions.'''

# Step 1: Summarize
def summarize(text):
    prompt = (
        "Summarize the passage in 2-3 sentences for a business audience.\n"
        "Passage:\n"
        f"{text}"
    )
    return LLM.chat(prompt).text

# Step 2: Critique
def critique(summary, text):
    prompt = (
        "Critique the summary for specificity, accuracy, and completeness.\n"
        "Then give two actionable suggestions to improve it.\n\n"
        f"Summary:\n{summary}\n\n"
        f"Reference Passage:\n{text}"
    )
    return LLM.chat(prompt).text

# Step 3: Improve (fixed version)
def improve(summary, feedback, text):
    prompt = (
        "Produce the final improved 2–3 sentence summary. "
        "Label it exactly as: Improved Summary:\n"
        "Include one mitigation technique and one evaluation best practice.\n\n"
        f"Original Summary:\n{summary}\n\n"
        f"Feedback (use this to improve):\n{feedback}\n\n"
        f"Reference Passage:\n{text}"
    )
    return LLM.chat(prompt).text

# Run the 3-step reflection chain
s1 = summarize(passage)
c1 = critique(s1, passage)
s2 = improve(s1, c1, passage)

# Safety check to ensure label
if "Improved Summary:" not in s2:
    s2 = "Improved Summary:\n" + s2.strip()

# Print results
print("Initial Summary:\n", s1, "\n")
print("Critique:\n", c1, "\n")
print(s2)

Initial Summary:
 Summary: LLMs can hallucinate; clear benchmarks and tool use reduce risk. 

Critique:
 Critique: Add one concrete example and an evaluation best practice. 

Improved Summary:
Critique: Add one concrete example and an evaluation best practice.
